In [ ]:
!pip install -q loralib

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import loralib as lora
import torchvision.models as models
import matplotlib.pyplot as plt

from torchvision.datasets import Food101
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights, EfficientNet_V2_S_Weights

In [ ]:
def replace_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new_module)

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

def get_food101_loaders(transform, batch_size=256, num_workers=2):
    train_dataset = Food101(
        root="data",
        split="train",
        download=True,
        transform=transform
    )

    val_dataset = Food101(
        root="data",
        split="test",
        download=True,
        transform=transform
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_dataloader, val_dataloader

In [ ]:
def build_resnet18_lora(num_classes=101, r=8, alpha=16):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Replace final classifier with a NORMAL trainable head
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # Apply LoRA to selected conv layers in the backbone
    target_layers = [
        "layer1.0.conv1", "layer1.0.conv2",
        "layer1.1.conv1", "layer1.1.conv2",
        "layer2.0.conv1", "layer2.0.conv2",
        "layer2.1.conv1", "layer2.1.conv2",
        "layer3.0.conv1", "layer3.0.conv2",
        "layer3.1.conv1", "layer3.1.conv2",
        "layer4.0.conv1", "layer4.0.conv2",
        "layer4.1.conv1", "layer4.1.conv2",
    ]

    for name, module in list(model.named_modules()):
        if name in target_layers and isinstance(module, nn.Conv2d):
            new_layer = lora.Conv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=module.kernel_size,
                stride=module.stride,
                padding=module.padding,
                dilation=module.dilation,
                groups=module.groups,
                bias=(module.bias is not None),
                r=r,
                lora_alpha=alpha
            )
            new_layer.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_layer.bias.data.copy_(module.bias.data)

            replace_module_by_name(model, name, new_layer)

    # Official Microsoft step: freeze everything except LoRA params
    lora.mark_only_lora_as_trainable(model)

    # IMPORTANT FIX:
    # keep classifier trainable for Food101
    for p in model.fc.parameters():
        p.requires_grad = True

    return model

In [ ]:
weights = ResNet18_Weights.DEFAULT
transform = weights.transforms()

train_dataloader, val_dataloader = get_food101_loaders(
    transform=transform,
    batch_size=256,
    num_workers=2
)

model = build_resnet18_lora(num_classes=101, r=8, alpha=16)
device = get_device()
model.to(device)

criterion = nn.CrossEntropyLoss()
training_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(training_params, lr=1e-3)

print("ResNet18 LoRA trainable params:", count_trainable_params(model))

ckpt_path = "model_weights/checkpoint_resnet18_lora.pt"
final_ckpt_path = "model_weights/final_checkpoint_resnet18_lora.pt"

train_losses = []
val_losses = []
train_accs = []
val_accs = []

if os.path.exists(ckpt_path):
    print("Loading checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"], strict=False)
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    best_val_acc = checkpoint["best_val_acc"]
    train_accs = checkpoint.get("train_accs", [])
    val_accs = checkpoint.get("val_accs", [])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    curr_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {curr_epoch} with best validation accuracy {best_val_acc:.4f}")
else:
    curr_epoch = 0
    best_val_acc = 0.0
    print("No checkpoint found, starting training from scratch")

100%|██████████| 5.00G/5.00G [03:14<00:00, 25.7MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 132MB/s]


AssertionError: 

In [ ]:
while curr_epoch < 8:
    train_loss, train_acc = train(model, train_dataloader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_dataloader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": curr_epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "train_accs": train_accs,
            "val_accs": val_accs,
            "train_losses": train_losses,
            "val_losses": val_losses
        }, ckpt_path)

    print(f"Epoch {curr_epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
    curr_epoch += 1

print("Training complete!")
torch.save({
    "epoch": curr_epoch - 1,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "best_val_acc": best_val_acc,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "train_losses": train_losses,
    "val_losses": val_losses
}, final_ckpt_path)

In [ ]:
if os.path.exists(final_ckpt_path):
    checkpoint = torch.load(final_ckpt_path, map_location=device)
    train_accs = checkpoint.get("train_accs", [])
    val_accs = checkpoint.get("val_accs", [])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    print(f"Best validation accuracy: {checkpoint['best_val_acc']:.4f}")

    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('ResNet18 LoRA: Training and Validation Loss')
    plt.legend()
    plt.show()

    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(val_accs, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('ResNet18 LoRA: Training and Validation Accuracy')
    plt.legend()
    plt.show()
else:
    print("No final checkpoint found")

In [ ]:
def build_efficientnet_v2_s_lora(num_classes=101, r=8, alpha=16):
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

    # Replace final classifier with a NORMAL trainable head
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Apply LoRA only to standard convs
    # skip grouped/depthwise convs because official loralib Conv2d is problematic for groups > 1
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Conv2d) and name.startswith("features"):
            if module.groups != 1:
                continue

            new_layer = lora.Conv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=module.kernel_size,
                stride=module.stride,
                padding=module.padding,
                dilation=module.dilation,
                groups=module.groups,
                bias=(module.bias is not None),
                r=r,
                lora_alpha=alpha
            )
            new_layer.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_layer.bias.data.copy_(module.bias.data)

            replace_module_by_name(model, name, new_layer)

    # Official Microsoft step
    lora.mark_only_lora_as_trainable(model)

    # IMPORTANT FIX:
    # keep classifier trainable
    for p in model.classifier[1].parameters():
        p.requires_grad = True

    return model

In [ ]:
def build_efficientnet_v2_s_lora(num_classes=101, r=8, alpha=16):
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

    # Replace final classifier with a NORMAL trainable head
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Apply LoRA only to standard convs
    # skip grouped/depthwise convs because official loralib Conv2d is problematic for groups > 1
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Conv2d) and name.startswith("features"):
            if module.groups != 1:
                continue

            new_layer = lora.Conv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=module.kernel_size,
                stride=module.stride,
                padding=module.padding,
                dilation=module.dilation,
                groups=module.groups,
                bias=(module.bias is not None),
                r=r,
                lora_alpha=alpha
            )
            new_layer.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_layer.bias.data.copy_(module.bias.data)

            replace_module_by_name(model, name, new_layer)

    # Official Microsoft step
    lora.mark_only_lora_as_trainable(model)

    # IMPORTANT FIX:
    # keep classifier trainable
    for p in model.classifier[1].parameters():
        p.requires_grad = True

    return model

In [ ]:
weights = EfficientNet_V2_S_Weights.DEFAULT
transform = weights.transforms()

train_dataloader, val_dataloader = get_food101_loaders(
    transform=transform,
    batch_size=256,
    num_workers=2
)

model = build_efficientnet_v2_s_lora(num_classes=101, r=8, alpha=16)
device = get_device()
model.to(device)

criterion = nn.CrossEntropyLoss()
training_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(training_params, lr=1e-3)

print("EfficientNetV2-S LoRA trainable params:", count_trainable_params(model))

ckpt_path = "model_weights/checkpoint_efficientnet_lora.pt"
final_ckpt_path = "model_weights/final_checkpoint_efficientnet_lora.pt"

train_losses = []
val_losses = []
train_accs = []
val_accs = []

if os.path.exists(ckpt_path):
    print("Loading checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"], strict=False)
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    best_val_acc = checkpoint["best_val_acc"]
    train_accs = checkpoint.get("train_accs", [])
    val_accs = checkpoint.get("val_accs", [])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    curr_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {curr_epoch} with best validation accuracy {best_val_acc:.4f}")
else:
    curr_epoch = 0
    best_val_acc = 0.0
    print("No checkpoint found, starting training from scratch")

In [ ]:
while curr_epoch < 8:
    train_loss, train_acc = train(model, train_dataloader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_dataloader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": curr_epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "train_accs": train_accs,
            "val_accs": val_accs,
            "train_losses": train_losses,
            "val_losses": val_losses
        }, ckpt_path)

    print(f"Epoch {curr_epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
    curr_epoch += 1

print("Training complete!")
torch.save({
    "epoch": curr_epoch - 1,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "best_val_acc": best_val_acc,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "train_losses": train_losses,
    "val_losses": val_losses
}, final_ckpt_path)

NameError: name 'curr_epoch' is not defined

In [ ]:
if os.path.exists(final_ckpt_path):
    checkpoint = torch.load(final_ckpt_path, map_location=device)
    train_accs = checkpoint.get("train_accs", [])
    val_accs = checkpoint.get("val_accs", [])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    print(f"Best validation accuracy: {checkpoint['best_val_acc']:.4f}")

    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('EfficientNetV2-S LoRA: Training and Validation Loss')
    plt.legend()
    plt.show()

    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(val_accs, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('EfficientNetV2-S LoRA: Training and Validation Accuracy')
    plt.legend()
    plt.show()
else:
    print("No final checkpoint found")